In [11]:
import json
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [12]:
from datetime import datetime

import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm

from src.analytics import VarianceCalculator
from src.core.schemas import OptionChain
from src.data.wrds_loader import WRDSOptionLoader

## Test de _interpolate_constant_maturity

In [13]:
print(VarianceCalculator._interpolate_constant_maturity(0.04, 23, 0.0625, 37))

0.053875


## Test de _integrate_variance

In [14]:
s0 = 4000.0
r = 0.05
sigma = 0.20
days = 30
t = days / 365
f = s0 * np.exp(r * t)

In [15]:
k = np.arange(2000, 8001, 5)
d1 = (np.log(s0 / k) + (r + sigma**2 / 2) * t) / (sigma * np.sqrt(t))
d2 = d1 - sigma * np.sqrt(t)
call = s0 * norm.cdf(d1) - k * np.exp(-r * t) * norm.cdf(d2)
put = k * np.exp(-r * t) * norm.cdf(-d2) - s0 * norm.cdf(-d1)
price = np.where(k < f, put, call)

In [16]:
otm = pd.DataFrame({"strike": k, "option_type": np.where(k < f, "P", "C"), "bid": price, "ask": price, "mid_price": price})
chain = OptionChain(
    date=datetime(2026, 11, 1), maturity_days=days, underlying_price=s0, risk_free_rate=r, forward_price=f, options_data=otm
)

In [17]:
print(VarianceCalculator._integrate_variance(otm, chain, is_svix=False))  # attendu ≈ 0.0400
print(VarianceCalculator._integrate_variance(otm, chain, is_svix=True))  # attendu ≈ 0.0401

0.04000052204544159
0.04006634300452335


## Test de _isolate_otm_options

In [18]:
df = pd.read_csv("../tests/data/dummy_spx_chain.csv")
with open("../tests/data/dummy_spx_context.json") as file:
    ctx = json.load(file)

ctx["date"] = datetime.fromisoformat(ctx["date"])
dummy_chain = OptionChain(**ctx, options_data=df)

otm_dummy = VarianceCalculator._isolate_otm_options(dummy_chain)
print(len(df), "options avant,", len(otm_dummy), "après")
otm_dummy

64 options avant, 32 après


,strike,option_type,bid,ask,mid_price
0,3600.0,P,0.52,0.53,0.53
1,3625.0,P,0.79,0.81,0.80
2,3650.0,P,1.17,1.20,1.19
3,3675.0,P,1.72,1.76,1.74
4,3700.0,P,2.48,2.53,2.50
5,3725.0,P,3.51,3.58,3.55
6,3750.0,P,4.90,5.00,4.95
7,3775.0,P,6.74,6.87,6.81
8,3800.0,P,9.12,9.30,9.21
9,3825.0,P,12.16,12.40,12.28


## Test de compute_metrics (pipeline complet, faux marché)

In [19]:
def make_bs_chain(days):
    t = days / 365
    fwd = s0 * np.exp(r * t)
    k = np.arange(2000, 8001, 5).astype(float)
    d1 = (np.log(s0 / k) + (r + sigma**2 / 2) * t) / (sigma * np.sqrt(t))
    d2 = d1 - sigma * np.sqrt(t)
    call = s0 * norm.cdf(d1) - k * np.exp(-r * t) * norm.cdf(d2)
    put = k * np.exp(-r * t) * norm.cdf(-d2) - s0 * norm.cdf(-d1)

    options = pd.DataFrame(
        {
            "strike": np.concatenate([k, k]),
            "option_type": ["C"] * len(k) + ["P"] * len(k),
            "mid_price": np.concatenate([call, put]),
        }
    )
    options["bid"] = options["mid_price"]
    options["ask"] = options["mid_price"]
    return OptionChain(datetime(2026, 11, 1), days, s0, r, fwd, options)


metrics = VarianceCalculator().compute_metrics([make_bs_chain(23), make_bs_chain(37)])
metrics

VolatilityMetrics(date=datetime.datetime(2026, 11, 1, 0, 0), maturity_days=30.0, is_interpolated=True, underlying_price=4000.0, risk_free_rate=0.05, dividend_yield=0.0, svix_annualized=0.20017078852549516, vix_annualized=0.19999731305010798, min_strike_used=2000.0, max_strike_used=8000.0, integration_points=1201)

## Pipeline complet avec les données de Nizar (WRDS)

In [20]:
chains = WRDSOptionLoader().get_interpolation_chains(datetime(2025, 3, 3), target_days=30)
metrics = VarianceCalculator().compute_metrics(chains)

vix_cboe = yf.download("^VIX", start="2025-03-03", end="2025-03-04")["Close"]

print("Échéances :", [c.maturity_days for c in chains])
print(f"Notre VIX : {metrics.vix_annualized:.2%}")
print(f"VIX CBOE  : {float(vix_cboe.iloc[0, 0]) / 100:.2%}")
print(f"Notre SVIX: {metrics.svix_annualized:.2%}")

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


[*********************100%***********************]  1 of 1 completed

Échéances : [29.0, 30.0]
Notre VIX : 22.13%
VIX CBOE  : 22.78%
Notre SVIX: 20.68%
